# 03 — Validation on measured PV data (DKASC Alice Springs)

Checks whether the algorithm ranking and the effect of Bayesian Optimization observed on the PVGIS-simulated data transfer to **measured** output of a real PV system: DKASC Alice Springs, Site 33 (REC, 5.5 kW, poly-Si, fixed, installed 2012), years 2014–2016.

Pipeline identical to notebook 02: hourly means, 70/15/15 random split, clipping and MinMax fitted on training rows, BO with 10 + 50 evaluations (objective −MAE on validation, seed-42 split), same search spaces, 5 seeds (42–46).

**Data are not included** (DKASC terms of use). Download instructions are in the README.

In [ ]:
# ============================================================
# CONFIGURATION: set DATA_DIR before running anything else
# ============================================================
import os

DATA_DIR = 'data'   # local run: folder that holds the input CSV and all outputs

try:                # Google Colab: mount Drive and point DATA_DIR to your Drive folder
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/<your-folder>'   # <-- change to your own Drive folder
except ImportError:
    pass

for sub in ['', 'tuning', 'stacking', 'multiseed', 'measured']:
    os.makedirs(os.path.join(DATA_DIR, sub), exist_ok=True)
print('DATA_DIR =', os.path.abspath(DATA_DIR))

In [ ]:
!pip install -q bayesian-optimization

## 1. Load, resample to hourly and filter

In [ ]:
import os, time, pickle, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

MEAS_DIR  = f'{DATA_DIR}/measured'; os.makedirs(MEAS_DIR, exist_ok=True)
MEAS_PATH = f'{MEAS_DIR}/52-Site_DKA-M16_C-Phase.csv'   # DKASC Alice Springs, Site 33 (see README)

COL_TIME  = 'timestamp'
COL_POWER = 'Active_Power'
COL_GHI   = 'Global_Horizontal_Radiation'
COL_DIFF  = 'Diffuse_Horizontal_Radiation'
COL_TEMP  = 'Weather_Temperature_Celsius'
YEARS = (2014, 2016)
CAPACITY_KWP = 5.5
MIN_SAMPLES_PER_HOUR = 10        # at least 10 of 12 five-minute records per hour
IRR_MIN = 10                     # W/m2: remove night/twilight (sensor offset 0.4–3 W/m2 at night)

raw = pd.read_csv(MEAS_PATH, usecols=[COL_TIME, COL_POWER, COL_GHI, COL_DIFF, COL_TEMP],
                  parse_dates=[COL_TIME])
raw = raw[(raw[COL_TIME].dt.year >= YEARS[0]) & (raw[COL_TIME].dt.year <= YEARS[1])]
raw = raw.drop_duplicates(subset=COL_TIME).set_index(COL_TIME).sort_index()

g  = raw.resample('1h')
dm = g.mean()
dm['n_valid'] = g[COL_POWER].count()
dm = dm[dm['n_valid'] >= MIN_SAMPLES_PER_HOUR].drop(columns='n_valid')
dm = dm.rename(columns={COL_POWER: 'P_kW', COL_GHI: 'irr_total',
                        COL_DIFF: 'irr_diffuse', COL_TEMP: 'temp_air'}).dropna()
dm = dm[(dm['irr_total'] >= IRR_MIN) & (dm['P_kW'] >= 0)].copy()

dm['year']      = dm.index.year
dm['hour']      = dm.index.hour
dm['month']     = dm.index.month
dm['hour_sin']  = np.sin(2*np.pi*dm['hour']/24);  dm['hour_cos']  = np.cos(2*np.pi*dm['hour']/24)
dm['month_sin'] = np.sin(2*np.pi*dm['month']/12); dm['month_cos'] = np.cos(2*np.pi*dm['month']/12)
dm = dm[dm['hour'].between(5, 20)].reset_index(drop=True)   # removes isolated night-time sensor glitches

FEATS_M = ['irr_total','irr_diffuse','temp_air','hour','month','hour_sin','hour_cos','month_sin','month_cos']
CLIP_M  = ['irr_total','irr_diffuse','temp_air']

print('Data ready:', dm.shape, '| period', YEARS)
print(dm.groupby('year').size().rename('daytime_hours_per_year'))
print(dm[['P_kW','irr_total','irr_diffuse','temp_air']].describe().round(3))
print('Share of rows with P_kW == 0:', round((dm['P_kW'] == 0).mean(), 3))
print(dm.groupby('hour').size().to_string())
print(dm.shape, '| min temp:', round(float(dm['temp_air'].min()), 2),
      '| rows below 0 °C:', int((dm['temp_air'] < 0).sum()))

## 2. Bayesian Optimization (once, on the seed-42 split)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from bayes_opt import BayesianOptimization

SEEDS_M = [42, 43, 44, 45, 46]
N_INIT_M, N_ITER_M = 10, 50

def prepare_m(tr_idx, te_idx):
    """1st–99th percentile clipping and MinMax scaling fitted on the training rows only."""
    Xtr = dm.loc[tr_idx, FEATS_M].copy(); Xte = dm.loc[te_idx, FEATS_M].copy()
    ytr = dm.loc[tr_idx, 'P_kW'].values.reshape(-1, 1)
    yte = dm.loc[te_idx, 'P_kW'].values.reshape(-1, 1)
    for c in CLIP_M:
        lo, hi = Xtr[c].quantile(0.01), Xtr[c].quantile(0.99)
        Xtr[c] = Xtr[c].clip(lo, hi); Xte[c] = Xte[c].clip(lo, hi)
    sc  = MinMaxScaler().fit(Xtr[CLIP_M])
    Xtr[CLIP_M] = sc.transform(Xtr[CLIP_M]); Xte[CLIP_M] = sc.transform(Xte[CLIP_M])
    ysc = MinMaxScaler().fit(ytr)
    return Xtr, ysc.transform(ytr).ravel(), Xte, ysc.transform(yte).ravel(), float(ysc.data_range_[0])

def mfull(y_true, y_pred, p_range):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2_score(y_true, y_pred),
            'MAE_kW': mae*p_range, 'RMSE_kW': rmse*p_range,
            'nMAE_%': mae*p_range/CAPACITY_KWP*100, 'nRMSE_%': rmse*p_range/CAPACITY_KWP*100}

# --- model builders (same as notebook 02) ---
def b_rf(p, s):
    return RandomForestRegressor(n_estimators=int(p['n_estimators']), max_depth=int(p['max_depth']),
        min_samples_split=int(p['min_samples_split']), min_samples_leaf=int(p['min_samples_leaf']),
        max_features=max(0.1, min(1.0, p['max_features'])), random_state=s, n_jobs=-1)
def b_xgb(p, s):
    return XGBRegressor(n_estimators=int(p['n_estimators']), max_depth=int(p['max_depth']),
        learning_rate=p['learning_rate'], subsample=p['subsample'], colsample_bytree=p['colsample_bytree'],
        gamma=p['gamma'], random_state=s, tree_method='hist', device='cpu', verbosity=0)
def b_lgbm(p, s):
    return LGBMRegressor(n_estimators=int(p['n_estimators']), max_depth=int(p['max_depth']),
        learning_rate=p['learning_rate'], num_leaves=int(p['num_leaves']), subsample=p['subsample'],
        colsample_bytree=p['colsample_bytree'], random_state=s, n_jobs=-1, verbose=-1)
def b_ada(p, s):
    return AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=int(p['max_depth'])),
        n_estimators=int(p['n_estimators']), learning_rate=p['learning_rate'], random_state=s)

BUILD = {'Random Forest': b_rf, 'XGBoost': b_xgb, 'LightGBM': b_lgbm, 'AdaBoost': b_ada}
SPACES = {   # identical to the search spaces of notebook 02
    'Random Forest': {'n_estimators': (50, 300), 'max_depth': (3, 20), 'min_samples_split': (2, 20),
                      'min_samples_leaf': (1, 10), 'max_features': (0.3, 1.0)},
    'XGBoost':  {'n_estimators': (50, 300), 'max_depth': (3, 10), 'learning_rate': (0.01, 0.3),
                 'subsample': (0.5, 1.0), 'colsample_bytree': (0.5, 1.0), 'gamma': (0, 5)},
    'LightGBM': {'n_estimators': (50, 300), 'max_depth': (3, 15), 'learning_rate': (0.01, 0.3),
                 'num_leaves': (20, 150), 'subsample': (0.5, 1.0), 'colsample_bytree': (0.5, 1.0)},
    'AdaBoost': {'n_estimators': (50, 300), 'learning_rate': (0.01, 1.0), 'max_depth': (1, 8)},
}

# seed-42 split -> train / val / test (BO uses train & val only)
_idx = dm.index.values
_tr, _tmp = train_test_split(_idx, test_size=0.30, random_state=42)
_va, _te  = train_test_split(_tmp, test_size=0.50, random_state=42)
Xtr_b, ytr_b, Xva_b, yva_b, _ = prepare_m(_tr, _va)

BO_PATH = f'{MEAS_DIR}/bo_params_measured.pkl'
if os.path.exists(BO_PATH):
    BO_M = pickle.load(open(BO_PATH, 'rb')); print('BO results loaded from file.')
else:
    BO_M = {}
    for name in BUILD:
        def obj(**p):
            m = BUILD[name](p, 42); m.fit(Xtr_b, ytr_b)
            return -mean_absolute_error(yva_b, m.predict(Xva_b))
        opt = BayesianOptimization(obj, SPACES[name], random_state=42, verbose=0)
        t0 = time.time(); opt.maximize(init_points=N_INIT_M, n_iter=N_ITER_M)
        BO_M[name] = {'params': opt.max['params'], 'cost_s': time.time() - t0}
        print(f'{name:<14} BO {BO_M[name]["cost_s"]:.0f} s | -MAE val = {opt.max["target"]:.6f}')
        pickle.dump(BO_M, open(BO_PATH, 'wb'))

## 3. Five-seed evaluation, tuning effect and rank agreement with PVGIS results

In [ ]:
from scipy.stats import spearmanr

def models_m(seed):
    out = [('Linear Regression', 'Conventional', LinearRegression()),
           ('Decision Tree', 'Conventional', DecisionTreeRegressor(random_state=seed)),
           ('Random Forest', 'Baseline', RandomForestRegressor(n_estimators=100, random_state=seed, n_jobs=-1)),
           ('XGBoost', 'Baseline', XGBRegressor(n_estimators=100, random_state=seed, tree_method='hist', device='cpu', verbosity=0)),
           ('LightGBM', 'Baseline', LGBMRegressor(n_estimators=100, random_state=seed, n_jobs=-1, verbose=-1)),
           ('AdaBoost', 'Baseline', AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=4), n_estimators=100, random_state=seed))]
    for n in BUILD:
        out.append((n, 'Tuned', BUILD[n](BO_M[n]['params'], seed)))
    out.append(('Stacking', 'Stacking', StackingRegressor(
        estimators=[('rf', BUILD['Random Forest'](BO_M['Random Forest']['params'], seed)),
                    ('xgb', BUILD['XGBoost'](BO_M['XGBoost']['params'], seed)),
                    ('lgbm', BUILD['LightGBM'](BO_M['LightGBM']['params'], seed)),
                    ('ada', BUILD['AdaBoost'](BO_M['AdaBoost']['params'], seed))],
        final_estimator=Ridge(alpha=1.0), cv=5, n_jobs=-1, passthrough=False)))
    return out

def run_seed_m(seed):
    tr, tmp = train_test_split(dm.index.values, test_size=0.30, random_state=seed)
    va, te  = train_test_split(tmp, test_size=0.50, random_state=seed)
    Xtr, ytr, Xte, yte, pr = prepare_m(tr, te)
    rows = []
    for name, stage, mdl in models_m(seed):
        t0 = time.time(); mdl.fit(Xtr, ytr); yp = mdl.predict(Xte)
        rows.append({'Seed': seed, 'Model': name, 'Stage': stage, **mfull(yte, yp, pr)})
        print(f'  seed {seed} | {name:<18} {stage:<12} MAE_kW={rows[-1]["MAE_kW"]:.5f} ({time.time()-t0:.0f}s)')
    return rows

MS_PATH = f'{MEAS_DIR}/ms_measured.pkl'
ms_meas = pickle.load(open(MS_PATH, 'rb')) if os.path.exists(MS_PATH) else {}
for s in SEEDS_M:
    if s in ms_meas: print(f'seed {s}: already done, skipped'); continue
    print(f'=== DKASC seed {s} ==='); ms_meas[s] = run_seed_m(s)
    pickle.dump(ms_meas, open(MS_PATH, 'wb'))

res = pd.DataFrame([r for s in SEEDS_M for r in ms_meas[s]])

# ---- main table: mean ± std over 5 seeds ----
summ = (res.groupby(['Model', 'Stage'], sort=False)
           .agg(MAE_kW=('MAE_kW', 'mean'), MAE_kW_sd=('MAE_kW', 'std'),
                RMSE_kW=('RMSE_kW', 'mean'), RMSE_kW_sd=('RMSE_kW', 'std'),
                R2=('R2', 'mean'), R2_sd=('R2', 'std'),
                nMAE_pct=('nMAE_%', 'mean'), nRMSE_pct=('nRMSE_%', 'mean')).reset_index())
summ['Rank'] = summ['MAE_kW'].rank().astype(int)
summ = summ.sort_values('Rank')
print('\nMEASURED DATA — DKASC Site 33 (5 seeds, test set)')
print(summ.to_string(index=False, float_format=lambda x: f'{x:.5f}'))
summ.to_csv(f'{MEAS_DIR}/table_measured.csv', index=False)

# ---- tuning effect: Tuned - Baseline (MAE_kW, per seed) ----
piv = res.pivot_table(index='Seed', columns=['Model', 'Stage'], values='MAE_kW')
print('\nTUNING EFFECT (Tuned - Baseline, W; negative = tuning improves)')
for m in BUILD:
    d = (piv[(m, 'Tuned')] - piv[(m, 'Baseline')]) * 1000
    print(f'  {m:<14} {d.mean():+8.3f} ± {d.std(ddof=1):.3f} W | improved in {(d < 0).sum()}/{len(d)} seeds')

# ---- stacking vs best single tuned model ----
best = min(BUILD, key=lambda m: piv[(m, 'Tuned')].mean())
d = (piv[('Stacking', 'Stacking')] - piv[(best, 'Tuned')]) * 1000
print(f'\nSTACKING vs {best} Tuned: difference {d.mean():+.3f} ± {d.std(ddof=1):.3f} W '
      f'(d_z={d.mean()/d.std(ddof=1):.2f}; negative = stacking better)')

# ---- rank agreement with the PVGIS results of notebook 02 ----
MAIN_PATH = f'{DATA_DIR}/multiseed/ms_random.pkl'
if os.path.exists(MAIN_PATH):
    ms_main = pickle.load(open(MAIN_PATH, 'rb'))
    mm = pd.DataFrame([r for s in ms_main for r in ms_main[s]])
    mm = mm[mm['Scope'] == 'All'].groupby(['Model', 'Stage'])['MAE_kW'].mean().rename('MAE_kW_PVGIS')
    cmp_ = summ.set_index(['Model', 'Stage'])[['MAE_kW', 'Rank']].join(mm, how='inner')
    cmp_['Rank_PVGIS'] = cmp_['MAE_kW_PVGIS'].rank().astype(int)
    rho, p = spearmanr(cmp_['MAE_kW'], cmp_['MAE_kW_PVGIS'])
    print('\nRANKING (measured vs PVGIS):')
    print(cmp_[['Rank', 'Rank_PVGIS']].sort_values('Rank').to_string())
    print(f'Spearman rho = {rho:.3f} (p = {p:.4f}, n = {len(cmp_)})')
else:
    print('\nRun notebook 02 first to compare the ranking with the PVGIS results.')